# MobileNetV2 Baseline Training — CIFAR-10

This notebook trains the FP32 MobileNetV2 baseline used for the quantization experiments.

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## 1. Model and dataset setup

In [ ]:
import torch
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v2(num_classes=10)
model = model.to(device)

print(model)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Training on:", device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

In [ ]:
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

## 2. Training configuration

In [ ]:
model = models.mobilenet_v2(num_classes=10)
model = model.to(device)

print("Fresh MobileNet-v2 created.")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

epochs = 20

best_test_accuracy = 0.0

## 3. Train the baseline model

In [ ]:
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    train_accuracy = 100 * correct / total

    # Evaluate on the test set
    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_accuracy = 100 * test_correct / test_total

    # Update learning rate
    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Accuracy: {test_accuracy:.2f}% | "
        f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}"
    )

    # Save only the best-performing model
    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy

        torch.save(
            model.state_dict(),
            "mobilenetv2_cifar10_best.pth"
        )

        print(
            f"Best model saved! "
            f"Test Accuracy: {best_test_accuracy:.2f}%"
        )

## 4. Load and save the best FP32 checkpoint

In [ ]:

best_model = models.mobilenet_v2(num_classes=10)
best_model.load_state_dict(
    torch.load(
        "mobilenetv2_cifar10_best.pth",
        map_location=device
    )
)

best_model = best_model.to(device)
best_model.eval()

print(f"Best test accuracy: {best_test_accuracy:.2f}%")

In [ ]:
import os

assignment_folder = "/content/drive/MyDrive/CS6886_MobileNetV2"

os.makedirs(assignment_folder, exist_ok=True)

print("Folder created:", assignment_folder)

In [ ]:
fp32_path = os.path.join(
    assignment_folder,
    "mobilenetv2_cifar10_fp32.pth"
)

torch.save(
    best_model.state_dict(),
    fp32_path
)

print("FP32 model saved successfully.")
print("Location:", fp32_path)

In [ ]:
import os

if os.path.exists(fp32_path):
    size_mb = os.path.getsize(fp32_path) / (1024 * 1024)

    print("File exists: Yes")
    print(f"Model size: {size_mb:.2f} MB")
else:
    print("File was not saved.")

## 5. Final baseline evaluation

In [ ]:
import torch

best_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

fp32_accuracy = 100 * correct / total

num_parameters = sum(
    parameter.numel()
    for parameter in best_model.parameters()
)

print(f"FP32 Test Accuracy: {fp32_accuracy:.2f}%")
print(f"Number of Parameters: {num_parameters:,}")
print("FP32 Model Size: 8.77 MB")